# GRPO Training with CodeArena RL Benchmark

This notebook demonstrates how to connect our custom `codearena-rl-benchmark` environment to HuggingFace's `trl.GRPOTrainer`.

In [ ]:
!pip install trl transformers datasets openenv-py httpx
!git clone https://github.com/havinashpatil/meta.git
!cd meta && pip install -r requirements.txt

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
import httpx

# Start the backend server in the background (Colab trick)
import subprocess
import time
subprocess.Popen(["uvicorn", "server.app:app", "--port", "7860", "--app-dir", "meta"])
time.sleep(5)  # Wait for server to start

In [ ]:
def codearena_reward_func(completions, prompts):
    """
    Reward function that queries the CodeArena OpenEnv server.
    For each proposed fix in `completions`, we step the environment.
    """
    rewards = []
    for completion in completions:
        # Clean the generated code
        proposed_fix = completion[0].get('content', '').strip()
        if proposed_fix.startswith('```python'):
            proposed_fix = proposed_fix[9:].replace('```', '').strip()
            
        try:
            # Step the environment
            res = httpx.post(
                "http://localhost:7860/step",
                json={"proposed_fix": proposed_fix},
                timeout=10.0
            )
            res.raise_for_status()
            reward = res.json().get('reward', 0.0)
            rewards.append(reward)
        except Exception as e:
            print(f"Env Error: {e}")
            rewards.append(0.0)
            
    return rewards

In [ ]:
# Load Model
model_name = "Qwen/Qwen2.5-Coder-1.5B"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Sample training dataset (prompts extracted from tasks)
# In a real setup, you'd reset the env for each prompt to get the initial buggy_code.
dataset = Dataset.from_dict({
    "prompt": [
        "Fix this Python code:\ndef average_list(numbers)\n    if length(numbers) == 0:\n        return 0\n    return sum(numbers) / length(numbers)"
    ]
})

# Initialize GRPO Trainer
training_args = GRPOConfig(
    output_dir="./codearena-grpo",
    learning_rate=1e-5,
    max_steps=50,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=codearena_reward_func,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()